In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
import pandas as pd
import time
import matplotlib.pyplot as plt # for making figures
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
from pprint import pprint

In [ ]:
torch.__version__

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
device

In [ ]:
# Get some names from https://github.com/MASTREX/List-of-Indian-Names

In [ ]:
import os
import requests

# Download only if file doesn't exist
if not os.path.exists('names-long.csv'):
    print("Downloading names dataset...")
    url = "data/Indian_Names.csv"
    response = requests.get(url)
    with open('names-long.csv', 'w') as f:
        f.write(response.text)
    print("Download complete!")
else:
    print("File already exists, skipping download.")

In [ ]:
!head names-long.csv

In [ ]:
!tail names-long.csv

In [ ]:
words = pd.read_csv('names-long.csv')["Name"]
words = words.str.lower()
words = words.str.strip()
words = words.str.replace(" ", "")

words = words[words.str.len() > 2]
words = words[words.str.len() < 10]

# Randomly shuffle the words
words = words.sample(frac=1).reset_index(drop=True)
words = words.tolist()

# Remove words having non alphabets
words = [word for word in words if word.isalpha()]
words[:10]


In [ ]:
len(words)

In [ ]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
pprint(itos)

In [ ]:
block_size = 5 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:]:
  
  #print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
# Move data to GPU

X = torch.tensor(X).to(device)
Y = torch.tensor(Y).to(device)

In [ ]:
X.shape, X.dtype, Y.shape, Y.dtype

In [ ]:
# Embedding layer for the context

emb_dim = 4
emb = torch.nn.Embedding(len(stoi), emb_dim)


In [ ]:
emb.weight

In [ ]:
emb.weight.shape

In [ ]:
# Function to visualize the embedding in 2d space

def plot_emb(emb, itos, ax=None):
    if ax is None:
        fig, ax = plt.subplots()
    
    # Get embedding weights and handle different dimensions
    weights = emb.weight.detach().cpu().numpy()
    
    if weights.shape[1] == 2:
        # For 2D embeddings, plot directly
        for i in range(len(itos)):
            x, y = weights[i]
            ax.scatter(x, y, color='k')
            ax.text(x + 0.05, y + 0.05, itos[i])
    elif weights.shape[1] > 2:
        # For higher dimensions, use PCA to reduce to 2D
        from sklearn.decomposition import PCA
        pca = PCA(n_components=2)
        weights_2d = pca.fit_transform(weights)
        
        for i in range(len(itos)):
            x, y = weights_2d[i]
            ax.scatter(x, y, color='k')
            ax.text(x + 0.05, y + 0.05, itos[i])
            
        # Add explained variance info to title
        explained_var = pca.explained_variance_ratio_
        ax.set_title(f'Embedding Visualization (PCA: {explained_var[0]:.2f}, {explained_var[1]:.2f} var explained)')
    else:
        # For 1D embeddings, create a scatter plot with y=0
        for i in range(len(itos)):
            x = weights[i, 0]
            y = 0
            ax.scatter(x, y, color='k')
            ax.text(x + 0.05, y + 0.05, itos[i])
        ax.set_ylabel('(1D embedding)')
    
    return ax

plot_emb(emb, itos)

In [ ]:
class NextChar(nn.Module):
  def __init__(self, block_size, vocab_size, emb_dim, hidden_size):
    super().__init__()
    self.emb = nn.Embedding(vocab_size, emb_dim)
    self.lin1 = nn.Linear(block_size * emb_dim, hidden_size)
    self.lin2 = nn.Linear(hidden_size, vocab_size)

  def forward(self, x):
    x = self.emb(x)
    x = x.view(x.shape[0], -1)
    x = torch.sin(self.lin1(x))
    x = self.lin2(x)
    return x


In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def print_model_summary(model):
    print("Model Summary:")
    for name, module in model.named_modules():
        if len(list(module.children())) == 0:
            params = sum(p.numel() for p in module.parameters())
            print(f"{name:<20} {params:,} parameters")
    print(f"\nTotal trainable parameters: {count_params(model):,}\n")


def train_model(model, X, Y, epochs=500, batch_size=1024, lr=1e-3, wd=1e-2, print_every=100):
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.CrossEntropyLoss()

    losses, times = [], []
    for epoch in range(epochs):
        start = time.time()
        total_loss, n_batches = 0, 0

        for i in range(0, X.shape[0], batch_size):
            x_batch = X[i:i+batch_size]
            y_batch = Y[i:i+batch_size]

            logits = model(x_batch)
            loss = loss_fn(logits, y_batch)

            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += loss.item()
            n_batches += 1

        avg_loss = total_loss / n_batches
        losses.append(avg_loss)
        times.append(time.time() - start)

        if epoch % print_every == 0:
            print(f"Epoch {epoch:4d} | Loss: {avg_loss:.4f} | Time: {times[-1]:.2f}s")

    return losses, times


In [ ]:
logits_demo = torch.tensor([0.1, 2.0, 2.5, -10.0])
prob_demo = F.softmax(logits_demo, dim=0)
itos_demo = {0: 'a', 1: 'b', 2: 'c', 3: 'd'}
counts = {ch: 0 for ch in itos_demo.values()}
for _ in range(1000):
    ix_demo = torch.distributions.categorical.Categorical(logits=logits_demo).sample()
    ch_demo = itos_demo[ix_demo.item()]
    counts[ch_demo] += 1
    #print(ch_demo)
print(counts)
print(prob_demo*1000)

In [ ]:
g = torch.Generator()
g.manual_seed(4000002)
def generate_name(model, itos, stoi, block_size, max_len=10):
    # Set model to evaluation mode
    model.eval()
    with torch.no_grad():
        context = [0] * block_size
        name = ''
        for i in range(max_len):
            x = torch.tensor(context).view(1, -1).to(device)
            y_pred = model(x)
            ix = torch.distributions.categorical.Categorical(logits=y_pred).sample().item()
            ch = itos[ix]
            if ch == '.':
                break
            name += ch
            context = context[1:] + [ix]
    # Set model back to training mode
    model.train()
    return name

In [ ]:
# Create enhanced model and show before training

model = NextChar(block_size, len(stoi), emb_dim, hidden_size=64).to(device)
model = torch.compile(model)

## Model parameters:
print_model_summary(model)

# Generate some names before training
print("Generated names before training:")
print("="*50)
for i in range(10):
    name = generate_name(model, itos, stoi, block_size)
    print(f"{i+1:2d}. {name}")

# Visualize embedding before training
print("\nEmbedding visualization before training:")
plot_emb(model.emb, itos)



In [ ]:
losses, times = train_model(model, X, Y, epochs=1200, batch_size=4096, lr=0.001, wd=0.01, print_every=200)  

In [ ]:
# Visualize the embedding post training
print("\nEmbedding visualization after training:")

plot_emb(model.emb, itos)

In [ ]:
# Generate some names after training
print("Generated names after training:")
print("="*50)
for i in range(10):
    name = generate_name(model, itos, stoi, block_size)
    print(f"{i+1:2d}. {name}")  


In [ ]:
class NextCharResidual(nn.Module):
    def __init__(self, block_size, vocab_size, emb_dim, hidden_size, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.fc_in = nn.Linear(block_size * emb_dim, hidden_size)
        
        # Residual MLP block
        self.block = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size),
        )
        
        self.bn_out = nn.BatchNorm1d(hidden_size)
        self.fc_out = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.emb(x)
        x = x.view(x.shape[0], -1)
        x = self.fc_in(x)
        residual = x
        x = self.block(x) + residual  # Residual connection
        x = torch.relu(self.bn_out(x))
        x = self.fc_out(x)
        return x


In [ ]:
model = NextCharResidual(block_size, len(stoi), emb_dim, hidden_size=128, dropout=0.2).to(device)


In [ ]:
print_model_summary(model)

print("="*60)
print("Before Training Samples:")
print("="*60)
model.eval()
with torch.no_grad():
    for i in range(10):
        name = generate_name(model, itos, stoi, block_size)
        print(f"{i+1:2d}. {name}")

In [ ]:
model.train()
losses, times = train_model(model, X, Y, epochs=1000, batch_size=2048, print_every=200)
print(f"Final loss: {losses[-1]:.4f}")

In [ ]:
print("\nEmbedding visualization after training:")
plot_emb(model.emb, itos)

In [ ]:
# Generate some names after training
print("Generated names after training:")
print("="*50)
for i in range(10):
    name = generate_name(model, itos, stoi, block_size)
    print(f"{i+1:2d}. {name}")



Tuning knobs

1. Embedding size
2. MLP 
3. Context length